# Contexto nocturno y fusión de probabilidades para DART + CRNN

**Pregunta.** ¿Se puede superar el 81,01 % de F1 macro del procedimiento anterior usando un solo LightGBM? Se conserva DART y se entrena una CRNN de 48 filtros, 48 unidades densas y GRU bidireccional de 32 unidades por dirección. Usa 15 épocas, Adam 0,001, lote 64, semilla 42, dropout 0,15/0,20 y la misma ponderación de clases con exponente 0,35.

**Qué aporta la literatura y cómo se prueba aquí**

- Vallat y Walker usan LightGBM con características espectrales y temporales suavizadas. Adaptamos sus ventanas de 7,5 min (triangular centrada) y 2 min (hacia atrás) como entradas de la CRNN, junto con espectro original y espectro normalizado por noche. Las estadísticas de cada noche usan exclusivamente su señal. Es inferencia retrospectiva sobre la noche completa. [YASA, eLife 2021](https://elifesciences.org/articles/70092)
- Phan y colaboradores combinan las decisiones de secuencias solapadas mediante agregación multiplicativa. Usamos tres posiciones por segmento, desplazadas 0, 5 y 10 épocas, y su media geométrica normalizada. Comparamos una y tres posiciones sobre los mismos pesos. [SeqSleepNet, IEEE TNSRE 2019](https://arxiv.org/abs/1809.10932)
- Kull y colaboradores describen una transformación lineal de log-probabilidades con salida softmax para calibración multiclase. Adaptamos esta idea para fusionar las salidas de DART, CRNN y HMM. La capa se ajusta en personas de validación interna, con ponderación de clases y personas. Aquí se evalúa clasificación mediante F1. [Calibración de Dirichlet, NeurIPS 2019](https://proceedings.neurips.cc/paper/2019/hash/8ca01ea920679a0fe3728441494041b9-Abstract.html)

**Cómo se calcula el F1.** En la primera ronda hay 34 personas: 20 para el ajuste interno, siete para elegir la combinación y siete para la prueba. Tras la elección, los modelos base de esa ronda usan las primeras 27 personas. La capa de fusión conserva el ajuste obtenido de predicciones internas fuera de muestra. Se repite el procedimiento rotando el grupo de prueba. Cada una de las 56 noches aparece exactamente una vez en el resultado global. Las cohortes se han usado antes para desarrollar modelos.

**Mejor configuración fija observada.** La fusión geométrica de 70 % DART, 15 % CRNN contextual y 15 % de esa misma CRNN tras el HMM alcanzó **81,33 % de F1 macro**, **87,06 % de exactitud** y **kappa 0,8212** en las 56 noches. La regla con selección interna obtuvo **81,24 %**, frente a **81,01 %** de la regla anterior. El intervalo pareado del 95 % para esa última diferencia va de −0,39 a +0,83 puntos de F1.

**Segunda iteración.** Elegir los pesos por cohorte dio **81,31 %** global. Frente a la mezcla geométrica fija, subió el F1 de Cassette de **77,22 % a 78,19 %** y bajó el de Telemetry de **80,58 % a 80,30 %**. El entrenamiento de la nueva red y el HMM tomó 29–31 segundos por ronda en CPU, con entradas preparadas. DART más esta rama sumaron 7 min 31 s en la primera ronda.

Las salidas quedan guardadas. Para reproducir desde los datos locales, ejecutar antes la [búsqueda anterior](busqueda_mezclas_dart_crnn_hmm.ipynb) para generar los DART y probabilidades de control de cada partición.

In [1]:
import os
import sys
from pathlib import Path
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'.dvc/config').is_file())
sys.path.insert(0, str(root))
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
from scripts.crnn_contexto_nocturno import entrenar
from scripts.buscar_fusion_crnn import (CLASES, CLAVES, SEMILLA, VALIDACION, carpeta_redes,
    nombre_particion, probabilidades, ruta_interna, preparar_dart_interno, preparar, f1_rapido)
from scripts.evaluar_v6_compacto import JUSTA
import json
import numpy as np
import pandas as pd
from scipy.special import softmax
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, cohen_kappa_score, confusion_matrix
from threadpoolctl import threadpool_limits
meta, dart, noches, _, _ = preparar()
previo = carpeta_redes(meta)
cache = entrenar(meta, noches)
internos = preparar_dart_interno(meta)
display(meta.groupby('cohorte').agg(personas=('subject_id','nunique'), noches=('record_id','nunique'), segmentos=('real','size')))

,personas,noches,segmentos
cohorte,,,
Cassette,12,12,32921
Telemetry,22,44,42691


## Técnicas y elección interna

Se fijaron cinco fusiones y una referencia anterior con porcentajes constantes. La nueva regla selecciona entre ellas con el F1 macro interno. Para puntuar las dos fusiones que aprenden coeficientes, se excluye a cada persona interna a su vez al estimar la capa. Así nadie elige la regla usando una predicción que se haya ajustado a su propia etiqueta. Todos los candidatos usan las mismas particiones. Las tablas de prueba incluyen cada técnica fija y la regla de elección.

La media conserva 75 % DART y 25 % HMM. La mezcla geométrica combina 70 % de log-probabilidad de DART, 15 % de CRNN y 15 % de HMM. La fusión aprendida recibe las tres distribuciones y ajusta una regresión logística multiclase con regularización L2. La última variante promedia esa salida con la media 75/25. El HMM siempre aprende las transiciones usando solo el entrenamiento correspondiente.

In [2]:
REFERENCIA = 'Amplia 75/25 anterior'
UNA = 'Contexto, una vista'
TRES = 'Contexto, tres vistas'
GEOM = 'Fusión geométrica'
LINEAL = 'Fusión aprendida'
MIXTA = 'Media + fusión aprendida'
NUEVA = 'Selección interna nueva'
ANTERIOR = 'Selección interna anterior'
metodos = [REFERENCIA, UNA, TRES, GEOM, LINEAL, MIXTA]

def entradas(frame, reservas, p_dart):
    particion = nombre_particion(reservas)
    ruta = cache/particion
    info = json.loads((ruta/'tiempo.json').read_text())
    assert set(frame.subject_id).isdisjoint(info['sujetos_train'])
    assert set(frame.subject_id).issubset(info['sujetos_reservados'])
    probs = pd.read_parquet(ruta/'prob.parquet').set_index(CLAVES, verify_integrity=True)
    probs = probs.loc[pd.MultiIndex.from_frame(frame[CLAVES])]
    una, crnn, hmm = [probs[[s+'_'+c for c in CLASES]].to_numpy() for s in ['una','red','hmm']]
    vieja = probabilidades(previo/particion/'Amplia/probabilidades.parquet', frame)
    logp = [np.log(np.maximum(p,1e-7)) for p in (p_dart,crnn,hmm)]
    caracteristicas = np.column_stack([v-v.mean(axis=1,keepdims=True) for v in logp])
    return {
        REFERENCIA: .75*p_dart+.25*vieja,
        UNA: .75*p_dart+.25*una,
        TRES: .75*p_dart+.25*hmm,
        GEOM: softmax(.70*logp[0]+.15*logp[1]+.15*logp[2],axis=1)
    }, caracteristicas

def ajustar_capa(x,y,sujetos):
    conteos = np.bincount(y,minlength=len(CLASES))
    assert (conteos>0).all()
    _, idx, num = np.unique(sujetos,return_inverse=True,return_counts=True)
    w = (conteos.mean()/conteos[y])**.35 * (num.mean()/num[idx])**.5
    modelo = LogisticRegression(C=.03, solver='lbfgs', max_iter=500, random_state=SEMILLA)
    with threadpool_limits(1):
        modelo.fit(x,y,sample_weight=w/w.mean())
    np.testing.assert_array_equal(modelo.classes_, np.arange(len(CLASES)))
    return modelo

def predecir(p):
    assert np.all(np.isfinite(p)) and np.allclose(p.sum(axis=1),1)
    return p.argmax(1)

resultados, decisiones, tiempos = [], [], []
for ronda, j in VALIDACION.items():
    val, te = meta[meta.ronda.eq(j)], meta[meta.ronda.eq(ronda)]
    assert set(val.subject_id).isdisjoint(te.subject_id)
    pval = probabilidades(ruta_interna(internos,ronda),val)
    pv, xv = entradas(val,tuple(sorted((ronda,j))),pval)
    y = pd.Categorical(val.real,categories=CLASES).codes.astype(int)
    sujetos = val.subject_id.to_numpy()
    pred_capa = np.zeros_like(pval)
    for persona in np.unique(sujetos):
        mask = sujetos==persona
        capa = ajustar_capa(xv[~mask],y[~mask],sujetos[~mask])
        pred_capa[mask] = capa.predict_proba(xv[mask])
    pv[LINEAL] = pred_capa
    pv[MIXTA] = .5*pv[TRES]+.5*pv[LINEAL]
    puntajes = {k:f1_rapido(y,predecir(pv[k])) for k in metodos}
    elegido = max(puntajes,key=puntajes.get)
    decisiones.append({'Ronda':ronda,'Validación interna':j,'Elegida':elegido,**puntajes})
    capa = ajustar_capa(xv,y,sujetos)
    # La configuración se congela antes de obtener las predicciones de prueba.
    pt, xt = entradas(te,(ronda,),dart[te.index])
    pt[LINEAL] = capa.predict_proba(xt)
    pt[MIXTA] = .5*pt[TRES]+.5*pt[LINEAL]
    g = te[CLAVES+['real','subject_id','cohorte','ronda','DART']].copy()
    for k in metodos:
        g[k] = CLASES[predecir(pt[k])]
    g[NUEVA] = g[elegido]
    resultados.append(g)
    destino = cache/nombre_particion((ronda,))
    np.savez(destino/'fusion_lineal.npz',coef=capa.coef_,intercept=capa.intercept_,
             clases=CLASES, elegida=elegido)
    info = json.loads((destino/'tiempo.json').read_text())
    tiempos.append([ronda,info['parametros'],info['ajuste_s'],info['inferencia_s'],te.record_id.nunique()])
oof = pd.concat(resultados,ignore_index=True)
anterior = pd.read_parquet(previo/'predicciones_seleccion.parquet')
llaves = CLAVES+['real','subject_id','cohorte','ronda','DART']
oof = oof.merge(anterior[llaves+['Selección interna']].rename(columns={'Selección interna':ANTERIOR}),
                on=llaves,how='left',validate='one_to_one')
assert len(oof)==len(meta)==75612 and oof.subject_id.nunique()==34 and oof.record_id.nunique()==56
assert not oof.duplicated(CLAVES).any() and not oof.isna().any().any()
oof.to_parquet(cache/'predicciones_externas.parquet',index=False)
display(pd.DataFrame(decisiones).round(4))

,Ronda,Validación interna,Elegida,Amplia 75/25 anterior,"Contexto, una vista","Contexto, tres vistas",Fusión geométrica,Fusión aprendida,Media + fusión aprendida
0,1,2,Fusión geométrica,0.7972,0.8071,0.8083,0.8159,0.7840,0.7993
1,2,1,Fusión geométrica,0.8293,0.8328,0.8329,0.8358,0.8028,0.8261
2,3,4,Fusión geométrica,0.7996,0.8013,0.8006,0.8083,0.7851,0.8001
3,4,3,Media + fusión aprendida,0.7998,0.8050,0.8046,0.7911,0.8042,0.8139
4,5,1,Amplia 75/25 anterior,0.8393,0.8281,0.8276,0.8314,0.8179,0.8300


## Métricas sobre todas las personas

El F1 global se calcula reuniendo las cinco pruebas. La comparación principal enfrenta la regla nueva con la regla anterior que obtuvo 81,01 %. Los intervalos remuestrean personas enteras, emparejando ambos resultados y conservando el tamaño de cada cohorte. Son intervalos orientativos para este conjunto de desarrollo.

In [3]:
def metricas(g, metodos_tabla):
    rows = []
    for k in metodos_tabla:
        y,p = g.real,g[k]
        rows.append([k,f1_score(y,p,labels=CLASES,average='macro',zero_division=0),
                     accuracy_score(y,p),cohen_kappa_score(y,p)])
    return pd.DataFrame(rows,columns=['Modelo','F1 macro','Exactitud','Kappa']).set_index('Modelo')
display(metricas(oof,['DART',ANTERIOR,*metodos,NUEVA]).round(5))
display(pd.concat({coh:metricas(g,['DART',ANTERIOR,NUEVA]) for coh,g in oof.groupby('cohorte')}).round(5))
display(pd.concat({i:metricas(g,[ANTERIOR,NUEVA]) for i,g in oof.groupby('ronda')}).round(5))
cm = confusion_matrix(oof.real,oof[NUEVA],labels=CLASES)
display(pd.DataFrame(cm,index=CLASES,columns=CLASES))
display(pd.DataFrame({k:f1_score(oof.real,oof[k],labels=CLASES,average=None) for k in [ANTERIOR,NUEVA]},index=CLASES).round(4))
rng = np.random.default_rng(SEMILLA)
conj = []
for _,g in oof.groupby('cohorte'):
    mats = np.array([[confusion_matrix(s.real,s[k],labels=CLASES) for k in [ANTERIOR,NUEVA]] for _,s in g.groupby('subject_id')])
    conj.append(mats[rng.integers(len(mats),size=(5000,len(mats)))].sum(axis=1))
cms = sum(conj)
den = cms.sum(axis=-1)+cms.sum(axis=-2)
f = np.divide(2*np.diagonal(cms,axis1=-2,axis2=-1),den,out=np.zeros(den.shape),where=den!=0).mean(axis=-1)
print('IC 95 % orientativo de la diferencia en puntos de F1:', np.round(100*np.quantile(f[:,1]-f[:,0],[.025,.975]),2))

,F1 macro,Exactitud,Kappa
Modelo,,,
DART,0.80098,0.85963,0.80740
Selección interna anterior,0.81005,0.86808,0.81766
Amplia 75/25 anterior,0.80907,0.86751,0.81708
"Contexto, una vista",0.81282,0.87021,0.82102
"Contexto, tres vistas",0.81279,0.87017,0.82095
Fusión geométrica,0.81329,0.87056,0.82116
Fusión aprendida,0.80456,0.86295,0.81167
Media + fusión aprendida,0.81245,0.87023,0.82128
Selección interna nueva,0.81240,0.87026,0.82091


F1 macro  Exactitud    Kappa
          Modelo                                                  
Cassette  DART                         0.77676    0.91443  0.82326
          Selección interna anterior   0.77116    0.91170  0.81538
          Selección interna nueva      0.77351    0.91525  0.82446
Telemetry DART                         0.78814    0.81736  0.74737
          Selección interna anterior   0.80170    0.83444  0.76729
          Selección interna nueva      0.80371    0.83556  0.76914

F1 macro  Exactitud    Kappa
  Modelo                                                  
1 Selección interna anterior   0.82949    0.88599  0.84140
  Selección interna nueva      0.83393    0.88850  0.84448
2 Selección interna anterior   0.80988    0.88600  0.83651
  Selección interna nueva      0.82467    0.89666  0.85179
3 Selección interna anterior   0.80853    0.84606  0.79026
  Selección interna nueva      0.80496    0.84691  0.79170
4 Selección interna anterior   0.81135    0.86497  0.81796
  Selección interna nueva      0.81323    0.86810  0.82283
5 Selección interna anterior   0.78702    0.85470  0.79502
  Selección interna nueva      0.77961    0.84710  0.78490

,N1,N2,N3,REM,W
N1,2829,929,19,362,720
N2,1117,21820,1946,641,504
N3,11,1058,6285,2,36
REM,333,796,10,8958,280
W,773,204,12,57,25910


,Selección interna anterior,Selección interna nueva
N1,0.5661,0.5702
N2,0.8543,0.8585
N3,0.8035,0.8025
REM,0.8773,0.8784
W,0.9491,0.9525


IC 95 % orientativo de la diferencia en puntos de F1: [-0.39  0.83]


## Tiempo de entrenamiento e inferencia

Se reutilizan los DART guardados. La suma de DART y CRNN corresponde al ajuste de los modelos base, con entradas preparadas. La selección interna requiere ajustes adicionales. La inferencia informada incluye tres vistas de la CRNN y el procesamiento de sus probabilidades para las variantes. La preparación desde EDF, DART y la capa lineal quedan fuera de ese cronómetro.

In [4]:
duracion = pd.DataFrame(tiempos,columns=['Ronda','Parámetros','Ajuste CRNN + HMM (s)','Inferencia variantes (s)','Noches de prueba'])
display(duracion.round(2))
s_dart = json.loads((JUSTA/'tiempos-ronda-1/mediciones.json').read_text())['ajuste_dart_s']
print(f'Primera ronda: DART {s_dart:.1f} s (medición anterior) + CRNN/HMM {tiempos[0][2]:.1f} s = {(s_dart+tiempos[0][2])/60:.2f} min.')

,Ronda,Parámetros,Ajuste CRNN + HMM (s),Inferencia variantes (s),Noches de prueba
0,1,43813,30.04,2.88,11
1,2,43813,29.44,2.81,11
2,3,43813,30.13,2.85,12
3,4,43813,29.70,2.79,12
4,5,43813,30.33,2.53,10


Primera ronda: DART 420.7 s (medición anterior) + CRNN/HMM 30.0 s = 7.51 min.


## Segunda iteración exploratoria: peso del HMM y frecuencia de las etapas

La primera serie mostró diferencias entre cohortes. En esta extensión se exploran dos reglas para adaptar la mezcla: una configuración común o una por cohorte con apoyo del conjunto interno completo. Se examinan 36 combinaciones de tres proporciones de DART, tres proporciones internas de HMM y cuatro correcciones suaves de la frecuencia de clase. Se ajusta todo dentro de cada ronda. Para la regla por cohorte, la función de elección promedia el F1 de la cohorte interna y el F1 de todas las personas internas, reduciendo la dependencia de los pocos sujetos de Cassette. Los candidatos empatan a favor de la primera opción en el orden del código. Esta extensión se planteó después de observar el primer informe de prueba y usa los mismos sujetos. Su resultado requiere confirmación independiente.

In [5]:
from itertools import product
GRUPO = 'Ajuste interno común'
COHORTE = 'Ajuste interno por cohorte'
rejilla = list(product([.75,.65,.85],[.5,0,1],[0,-.15,.15,.30]))

def fuentes(g,reservas,pd_art):
    carpeta = cache/nombre_particion(reservas)
    tabla = pd.read_parquet(carpeta/'prob.parquet').set_index(CLAVES, verify_integrity=True)
    tabla = tabla.loc[pd.MultiIndex.from_frame(g[CLAVES])]
    train = meta[~meta.ronda.isin(reservas)]
    assert set(train.subject_id).isdisjoint(g.subject_id)
    prior = train.real.value_counts(normalize=True).reindex(CLASES).to_numpy()
    partes = [pd_art]+[tabla[[pref+'_'+c for c in CLASES]].to_numpy() for pref in ['red','hmm']]
    return [np.log(np.maximum(a,1e-7)) for a in partes], np.log(prior)

def clasificar(fuentes_datos,config):
    (ld,lr,lh), logprior = fuentes_datos
    wd,h,b=config
    return (wd*ld+(1-wd)*((1-h)*lr+h*lh)-b*logprior).argmax(axis=1)

extension, opciones = [], []
for ronda,j in VALIDACION.items():
    val, te = meta[meta.ronda.eq(j)],meta[meta.ronda.eq(ronda)]
    xv = fuentes(val,tuple(sorted((ronda,j))),probabilidades(ruta_interna(internos,ronda),val))
    y = pd.Categorical(val.real,categories=CLASES).codes.astype(int)
    coh = val.cohorte.to_numpy()
    scores = {k:[] for k in ['Ambas',*sorted(set(coh))]}
    for config in rejilla:
        clasif = clasificar(xv,config)
        macro = f1_rapido(y,clasif)
        scores['Ambas'].append(macro)
        for k in sorted(set(coh)):
            mask=coh==k
            scores[k].append(.5*macro+.5*f1_rapido(y[mask],clasif[mask]))
    configs = {k:rejilla[int(np.argmax(v))] for k,v in scores.items()}
    for k,(wd,h,b) in configs.items():
        opciones.append([ronda,k,wd,h,b])
    xt = fuentes(te,(ronda,),dart[te.index])
    ext = te[CLAVES].copy()
    ext[GRUPO] = CLASES[clasificar(xt, configs['Ambas'])]
    for k,g in te.groupby('cohorte'):
        indices = te.index.get_indexer(g.index)
        ext.loc[g.index, COHORTE] = CLASES[clasificar(xt,configs[k])[indices]]
    extension.append(ext)
evaluacion = oof.merge(pd.concat(extension),on=CLAVES,how='left',validate='one_to_one')
assert len(evaluacion)==len(oof) and not evaluacion[[GRUPO,COHORTE]].isna().any().any()
evaluacion.to_parquet(cache/'predicciones_con_extension.parquet',index=False)
display(metricas(evaluacion,[ANTERIOR,NUEVA,GEOM,GRUPO,COHORTE]).round(5))
display(pd.concat({k:metricas(g,[GEOM,GRUPO,COHORTE]) for k,g in evaluacion.groupby('cohorte')}).round(5))
display(pd.DataFrame(opciones,columns=['Ronda','Grupo','Peso DART','Fracción HMM en la rama','Exponente de frecuencia']))

,F1 macro,Exactitud,Kappa
Modelo,,,
Selección interna anterior,0.81005,0.86808,0.81766
Selección interna nueva,0.81240,0.87026,0.82091
Fusión geométrica,0.81329,0.87056,0.82116
Ajuste interno común,0.81020,0.86663,0.81608
Ajuste interno por cohorte,0.81306,0.86980,0.82048


F1 macro  Exactitud    Kappa
          Modelo                                                  
Cassette  Fusión geométrica            0.77222    0.91364  0.82025
          Ajuste interno común         0.77352    0.91024  0.81426
          Ajuste interno por cohorte   0.78188    0.91616  0.82585
Telemetry Fusión geométrica            0.80582    0.83734  0.77128
          Ajuste interno común         0.80115    0.83301  0.76618
          Ajuste interno por cohorte   0.80302    0.83404  0.76768

,Ronda,Grupo,Peso DART,Fracción HMM en la rama,Exponente de frecuencia
0,1,Ambas,0.65,0.5,0.00
1,1,Cassette,0.65,0.0,-0.15
2,1,Telemetry,0.65,0.0,-0.15
3,2,Ambas,0.65,0.0,-0.15
4,2,Cassette,0.65,0.0,-0.15
5,2,Telemetry,0.65,0.0,-0.15
6,3,Ambas,0.75,1.0,0.30
7,3,Cassette,0.75,0.5,0.30
8,3,Telemetry,0.75,1.0,0.30
9,4,Ambas,0.85,0.0,0.15
